# Topic modeling (LDA) — exploratory pass

Explores what policy topics are discussed in state-parliament speeches, scoped to each state's
legislative period immediately before vs. after AfD entry. This does not measure morality or
politeness — the topic assigned to each speech is meant as a **control** variable for later
analysis (some topics may be more polarized/moralized than others, independent of AfD entry).

Runs two independent topic-count-selection methods and compares them: gensim LDA scored by c_v
coherence, and sklearn LDA scored by held-out log-likelihood (the latter per the practical guide
https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf, which
argues coherence is unreliable for tuning -- rather than pick a side, both run and get compared).

See `docs/superpowers/specs/2026-08-14-topic-modeling-lda-design.md` (local-only, not tracked in
git) for full design rationale.

In [1]:
import os
import sys
import time

import spacy
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(".."))
load_dotenv("../.env")

from topic_modeling_lib import (
    build_gensim_corpus,
    build_sklearn_corpus,
    gensim_coherence_scan,
    load_corpus,
    make_spacy_preprocessor,
    sklearn_loglikelihood_search,
    timed,
)

DATA_ROOT = os.environ["DATA_ROOT"]
timing_log = []

## Parameters

Start small (single state, sampled) to get a fast timing read before scaling up -- see the
"Timing" cell at the end for what that read implies about a full run.

In [2]:
STATES = ["by"]       # None = all 16 states
PRE_POST = None        # "pre", "post", or None for both
SAMPLE_N = 2000         # None = use every speech in scope
K_RANGE = [5, 10, 15, 20, 25, 30]
SKLEARN_N_ITER = 6      # None = try every k in K_RANGE

run_params = {
    "states": STATES,
    "pre_post": PRE_POST,
    "sample_n": SAMPLE_N,
    "preprocessing": "spacy_de_core_news_lg_v1",
}

In [3]:
with timed("load_corpus", log=timing_log):
    corpus_df = load_corpus(DATA_ROOT, states=STATES, pre_post=PRE_POST, sample_n=SAMPLE_N)
print(f"{len(corpus_df):,} documents loaded")

with timed("load spaCy model", log=timing_log):
    nlp = spacy.load("de_core_news_lg")
preprocess = make_spacy_preprocessor(nlp)

with timed("preprocess", log=timing_log):
    tokenized_docs = preprocess(corpus_df["text"].tolist())
print(f"Example tokens: {tokenized_docs[0][:10]}")

[load_corpus] 1.01s
2,000 documents loaded


[load spaCy model] 0.54s


[preprocess] 48.47s
Example tokens: ['verehrt', 'herr', 'präsident', 'lieb', 'kollegin', 'kollege', 'liebe', 'mitbürger', 'außerhalb', 'politikerblase']


## Vectorize and fit both models across K

Both methods share the same `tokenized_docs` and `K_RANGE`, but vectorize independently (gensim's
`Dictionary`/BoW vs. sklearn's `CountVectorizer`/doc-term matrix) since each library needs its own
input format. Results are cached under `measurement/run_history/topic_modeling/`, keyed by
`run_params` + K range + method -- rerunning this notebook with the same parameters loads from
disk instead of re-fitting.

In [4]:
with timed("build_gensim_corpus", log=timing_log):
    dictionary, gensim_corpus = build_gensim_corpus(tokenized_docs)

with timed("gensim_coherence_scan (all k)", log=timing_log):
    gensim_results = gensim_coherence_scan(
        tokenized_docs, dictionary, gensim_corpus, k_range=K_RANGE, params=run_params,
    )
gensim_results[["k", "coherence", "seconds"]]

[build_gensim_corpus] 0.15s


[gensim k=5] 5.09s, coherence=0.2957


[gensim k=10] 5.26s, coherence=0.2924


[gensim k=15] 5.14s, coherence=0.2921


[gensim k=20] 5.27s, coherence=0.2918


[gensim k=25] 5.35s, coherence=0.2892


[gensim k=30] 5.44s, coherence=0.2904
[gensim_coherence_scan (all k)] 31.57s


,k,coherence,seconds
0,5,0.295716,5.090255
1,10,0.292446,5.261957
2,15,0.292126,5.135339
3,20,0.291827,5.265707
4,25,0.289168,5.347914
5,30,0.290425,5.440563


In [5]:
with timed("build_sklearn_corpus", log=timing_log):
    vectorizer, dtm = build_sklearn_corpus(tokenized_docs)

with timed("sklearn_loglikelihood_search (all k)", log=timing_log):
    sklearn_results = sklearn_loglikelihood_search(
        dtm, k_range=K_RANGE, params=run_params, n_iter=SKLEARN_N_ITER,
    )
sklearn_results[["k", "log_likelihood", "seconds"]]

[build_sklearn_corpus] 0.06s


[sklearn k=10] 2.16s, log_likelihood=-711759.85


[sklearn k=20] 2.37s, log_likelihood=-810374.53


[sklearn k=25] 2.43s, log_likelihood=-860980.67


[sklearn k=15] 2.27s, log_likelihood=-764570.77


[sklearn k=5] 1.66s, log_likelihood=-643070.17


[sklearn k=30] 2.49s, log_likelihood=-908835.48
[sklearn_loglikelihood_search (all k)] 13.43s


,k,log_likelihood,seconds
0,10,-711759.845885,2.157679
1,20,-810374.529525,2.374958
2,25,-860980.665335,2.434070
3,15,-764570.774596,2.270925
4,5,-643070.167309,1.656985
5,30,-908835.476286,2.490304


## Compare the two methods' preferred K

In [6]:
best_gensim_k = int(gensim_results.loc[gensim_results["coherence"].idxmax(), "k"])
best_sklearn_k = int(sklearn_results.loc[sklearn_results["log_likelihood"].idxmax(), "k"])
print(f"gensim (c_v coherence) prefers k={best_gensim_k}")
print(f"sklearn (log-likelihood) prefers k={best_sklearn_k}")
print("Agreement" if best_gensim_k == best_sklearn_k else "Disagreement -- inspect both before picking K")

gensim (c_v coherence) prefers k=5
sklearn (log-likelihood) prefers k=5
Agreement


## Inspect topics for the chosen K

In [7]:
chosen_k = best_gensim_k  # change after inspecting the comparison above
chosen_model = gensim_results.set_index("k").loc[chosen_k, "model"]

for topic_id, terms in chosen_model.print_topics(num_words=10):
    print(f"Topic {topic_id}: {terms}\n")

Topic 0: 0.011*"herr" + 0.010*"bayern" + 0.008*"kollege" + 0.006*"sagen" + 0.005*"bayerisch" + 0.005*"mensch" + 0.005*"kollegin" + 0.004*"frau" + 0.004*"brauchen" + 0.004*"euro"

Topic 1: 0.009*"herr" + 0.008*"bayern" + 0.007*"bayerisch" + 0.007*"kollege" + 0.006*"antrag" + 0.006*"mensch" + 0.006*"sagen" + 0.005*"stehen" + 0.005*"kollegin" + 0.005*"frau"

Topic 2: 0.009*"kollege" + 0.007*"herr" + 0.007*"kollegin" + 0.007*"sagen" + 0.006*"antrag" + 0.006*"bayern" + 0.004*"wissen" + 0.004*"mensch" + 0.004*"unser" + 0.003*"thema"

Topic 3: 0.008*"herr" + 0.005*"bayern" + 0.005*"kollege" + 0.005*"sagen" + 0.005*"antrag" + 0.004*"mensch" + 0.004*"bayerisch" + 0.004*"kollegin" + 0.004*"wissen" + 0.003*"frau"

Topic 4: 0.012*"bayern" + 0.011*"herr" + 0.011*"kollege" + 0.006*"kollegin" + 0.005*"sagen" + 0.004*"brauchen" + 0.004*"antrag" + 0.004*"lieb" + 0.004*"bayerisch" + 0.004*"wichtig"



## Timing summary — for estimating full-corpus runtime

This ran on the sample size and state(s) set in the Parameters cell above. Multiply the
preprocess/vectorize/fit seconds-per-document by the full pre/post-AfD corpus size (see
`preprocessing/afd_period_window.py`'s printed document count) to estimate a full run's cost
before committing to it.

In [8]:
import pandas as pd

timing_df = pd.DataFrame(timing_log)
timing_df["seconds_per_doc"] = timing_df["seconds"] / len(corpus_df)
timing_df

,step,seconds,seconds_per_doc
0,load_corpus,1.010865,0.000505
1,load spaCy model,0.543784,0.000272
2,preprocess,48.472992,0.024236
3,build_gensim_corpus,0.150214,0.000075
4,gensim_coherence_scan (all k),31.574037,0.015787
5,build_sklearn_corpus,0.056374,0.000028
6,sklearn_loglikelihood_search (all k),13.432026,0.006716
